In [3]:
import pandas as pd
import numpy as np
import datetime
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [4]:
def calculate_nea_tariff_rate(monthly_units):
    """
    Returns the per-unit price (NPR) based on the total units 
    consumed so far during the current billing cycle (month).
    """
    if monthly_units <= 20:
        return 4.00   # Base rate for 15A meter standard
    elif monthly_units <= 30:
        return 6.50
    elif monthly_units <= 50:
        return 8.00
    elif monthly_units <= 150:
        return 9.50
    elif monthly_units <= 250:
        return 9.50
    else:
        return 11.00

In [8]:
# Generate 30 days of 15-minute interval data (4 intervals/hour * 24 hours * 30 days)
start_date = datetime.datetime(2026, 6, 1)
intervals = 4 * 24 * 30  
timestamps = [start_date + datetime.timedelta(minutes=15 * i) for i in range(intervals)]

data = []

# STATE MEMORY TRACKING VARIABLES
# These variables persist across loop cycles to maintain time-series consistency
cumulative_energy = 0.0
current_balance = np.random.uniform(50.0, 350.0)  # Random start balance for data variety
prev_load = np.random.uniform(0.3, 0.7)          # Seed initial previous load naturally
is_vacation_day = False

In [9]:
import pandas as pd
import numpy as np
import datetime

print("Generating long-term historical windowed dataset (3 months)...")

np.random.seed(42)

# Generate 90 days of data to establish healthy weekly/monthly rolling baselines
start_date = datetime.datetime(2026, 6, 1)
intervals = 4 * 24 * 90  # 15-minute steps over 90 days = 8,640 rows
timestamps = [start_date + datetime.timedelta(minutes=15 * i) for i in range(intervals)]

data = []
cumulative_energy = 0.0
prev_load = 0.5
is_vacation_day = False

for ts in timestamps:
    hour = ts.hour
    
    # Monthly reset for Nepal slab tariff calculation
    if ts.day == 1 and hour == 0 and ts.minute == 0:
        cumulative_energy = 0.0
        
    # Roll for vacation days
    if hour == 0 and ts.minute == 0:
        is_vacation_day = np.random.rand() < 0.04 
        
    # Baseline load shapes
    if is_vacation_day:
        base_load = np.random.uniform(0.05, 0.12)
    else:
        if 18 <= hour <= 22:
            base_load = np.random.uniform(2.5, 4.0)
        elif 7 <= hour <= 9:
            base_load = np.random.uniform(1.5, 2.5)
        else:
            base_load = np.random.uniform(0.3, 0.7)
            
    load_curr = max(0.02, base_load + np.random.normal(0, 0.08))
    
    # Injecting Anomaly (Theft Profile)
    theft_label = 0
    if not is_vacation_day and (18 <= hour <= 22) and (np.random.rand() < 0.01):
        load_curr = np.random.uniform(0.05, 0.15)
        theft_label = 1
        
    delta_load = load_curr - prev_load
    energy_delta = load_curr * 0.25
    cumulative_energy += energy_delta
    
    data.append([ts, hour, load_curr, cumulative_energy, delta_load, theft_label])
    prev_load = load_curr

# Create baseline DataFrame
columns = ['Timestamp', 'Hour_tod', 'Load_curr', 'Energy_cum', 'Delta_load', 'Ground_Truth_Theft']
df = pd.DataFrame(data, columns=columns)

# =====================================================================
# MACRO FEATURE ENGINEERING: THE WEEKLY & MONTHLY WINDOWING
# =====================================================================
# 1 day = 96 intervals (4 * 24). 7 days = 672 intervals. 30 days = 2880 intervals.
window_1_week = 4 * 24 * 7
window_1_month = 4 * 24 * 30

print("Calculating rolling historical macro parameters...")

# Calculate the rolling average consumption over the past week and month
df['Rolling_Avg_Week'] = df['Load_curr'].rolling(window=window_1_week, min_periods=1).mean()
df['Rolling_Avg_Month'] = df['Load_curr'].rolling(window=window_1_month, min_periods=1).mean()

# Calculate variance over the past week
df['Rolling_Std_Week'] = df['Load_curr'].rolling(window=window_1_week, min_periods=1).std().fillna(0.1)

# Calculate how many standard deviations the current load drops below the weekly average
# A normal family going out drops slightly. Theft causes a massive statistical plunge.
df['Z_Score_Deviation'] = (df['Rolling_Avg_Week'] - df['Load_curr']) / df['Rolling_Std_Week']

# Clean up data anomalies from initial empty window warmups
df['Rolling_Avg_Week'] = df['Rolling_Avg_Week'].round(3)
df['Rolling_Avg_Month'] = df['Rolling_Avg_Month'].round(3)
df['Z_Score_Deviation'] = df['Z_Score_Deviation'].round(3)

# Save the macro-enabled spreadsheet
df.to_csv('meter_windowed_training_data.csv', index=False)
print("Saved long-term feature dataset: 'meter_windowed_training_data.csv'")
print(df[['Timestamp', 'Load_curr', 'Rolling_Avg_Week', 'Z_Score_Deviation', 'Ground_Truth_Theft']].tail(10))

Generating long-term historical windowed dataset (3 months)...
Calculating rolling historical macro parameters...
Saved long-term feature dataset: 'meter_windowed_training_data.csv'
               Timestamp  Load_curr  Rolling_Avg_Week  Z_Score_Deviation  \
8630 2026-08-29 21:30:00   3.126421             1.255             -1.615   
8631 2026-08-29 21:45:00   2.962871             1.256             -1.472   
8632 2026-08-29 22:00:00   3.922969             1.262             -2.287   
8633 2026-08-29 22:15:00   3.683555             1.263             -2.076   
8634 2026-08-29 22:30:00   3.926520             1.264             -2.281   
8635 2026-08-29 22:45:00   3.293172             1.264             -1.738   
8636 2026-08-29 23:00:00   0.335163             1.264              0.795   
8637 2026-08-29 23:15:00   0.605974             1.264              0.563   
8638 2026-08-29 23:30:00   0.343867             1.264              0.788   
8639 2026-08-29 23:45:00   0.659271             1.264     